# Kybermart CV Demo — YOLO26x Fine-Tuning on Warehouse Box Dataset
Run this in Google Colab. YOLO26x is the largest variant — use an A100 if available; T4 will work but slower.

Dataset: exported from Roboflow via curl link.

In [ ]:
# 1. Confirm GPU
!nvidia-smi

In [ ]:
# 2. Install deps (upgrade to latest ultralytics for YOLO26 support)
!pip install -U ultralytics -q

In [ ]:
# 3. Download dataset from Roboflow (curl export link)
!mkdir -p /content/dataset
%cd /content/dataset
!curl -L "https://app.roboflow.com/ds/91AKDknuYm?key=tE84qyBySc" > roboflow.zip
!unzip -q roboflow.zip
!rm roboflow.zip
!cat data.yaml

In [ ]:
# 4. Train YOLO26x on the dataset
from ultralytics import YOLO

model = YOLO('yolo26x.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=1280,
    batch=8,          # x-variant is large; lower batch if you hit OOM
    name='kybermart_yolo26x',
    device=0,
    patience=20,
    augment=True,
    mosaic=1.0,
)
print('Best weights:', results.save_dir)

In [ ]:
# 5. Validate and check mAP
model_best = YOLO(str(results.save_dir) + '/weights/best.pt')
val = model_best.val(data='/content/dataset/data.yaml')
print(f'mAP50: {val.box.map50:.3f}')
print(f'mAP50-95: {val.box.map:.3f}')

In [ ]:
# 6. Download best.pt to your machine
from google.colab import files
files.download(str(results.save_dir) + '/weights/best.pt')
# Then put it in C:\Users\leodo\claude\kybermart\models\kybermart_yolo26x_best.pt
# and send me the file — I'll wire it into run_kybermart.py in place of the current public model.